# FLAG 2027 - Notebook 06: More training people (MAV-Celeb v1 + v3, approved by the organizers)

**Hypothesis:** same-gender error is stuck at ~39 because 70 training people are too few. MAV-Celeb v1 (70 people, English+Urdu) and v3 (50 people, English+German) give **120 more people** (no Bengali).

**Steps**
1. Load the official ECAPA voice model (`yangwang825/ecapa-tdnn-vox2`, same as the organizers) and **check** that our extraction reproduces the official challenge voice features.
2. Extract CLIP (faces), ECAPA and WavLM (voices) for MAV-Celeb v1+v3 train data. Saved to Drive; resumable.
3. Train our best system (K10: CLIP | ECAPA+WavLM) with: E0 challenge only / E1 + MAV-Celeb English clips / E2 + all MAV-Celeb clips. Validation always on held-out CHALLENGE people (same folds as before).
4. Build submission ZIPs for E1 and E2 (CodaBench shows whether the extra languages help Bengali).

**How to run:** T4 GPU, Runtime > Run all. First run about 1.5-2.5 hours (most of it feature extraction). If Colab disconnects: Run all again - finished parts are skipped.

**When done:** paste the **SUMMARY** to Claude.

In [ ]:
# Cell 1 - Drive, GPU, install SpeechBrain (for the official ECAPA model)
from google.colab import drive
drive.mount('/content/drive')
!pip -q install speechbrain
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev)

In [ ]:
# Cell 2 - paths, helpers, challenge features (same as Notebook 05)
import os, json, time, zipfile, shutil, copy
import numpy as np, pandas as pd
import librosa, soundfile as sf
from PIL import Image
from tqdm.auto import tqdm

BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
FEATDIR = os.path.join(PROJECT, 'features')
CACHE = os.path.join(PROJECT, 'models_cache')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
MAV_ZIPS = os.path.join(BASE, 'dataset')
MAV_LISTS = os.path.join(BASE, 'face_voice_association_splits')
LOCAL = '/content/flag_small'
os.makedirs(LOCAL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    flag = os.path.join(LOCAL, z + '.done')
    if not os.path.exists(flag):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(LOCAL, members=[m for m in zf.namelist() if m.endswith(('.csv', '.txt'))])
        open(flag, 'w').close()
INDEX = [os.path.join(r, f) for r, _, fs in os.walk(LOCAL) for f in fs if f.endswith(('.csv', '.txt'))]
def find_one(*parts):
    hits = [p for p in INDEX if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)

SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
FEAT = {}
Xv, y = read_feat(find_one('train_English_voices.csv'), 192)
FEAT['ecapa'] = {'train': Xv}
for s in SPLIT_NAMES[1:]:
    track, lang = s.rsplit('_', 1)
    FEAT['ecapa'][s] = read_feat(find_one('/' + track + '/', lang + '_test_voices.csv'), 192)[0]
for name in ['clip', 'wavlm_sv']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in SPLIT_NAMES}
y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y)
id_gender = {c: spk2g[cls2spk[c]] for c in ids}

def zip_index(zf):
    # map 'voices/...' or 'faces/...' (path from that folder on) -> zip member name
    m = {}
    for n in zf.namelist():
        for key in ('voices/', 'faces/', 'voice/', 'face/'):
            i = n.find(key)
            if i >= 0 and not n.endswith('/'):
                m[n[i:]] = n; break
    return m

SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)

In [ ]:
# Cell 3 - the three encoders (official ECAPA exactly as in the organizers' script, CLIP, WavLM)
try:
    from speechbrain.inference.interfaces import Pretrained
except ImportError:
    from speechbrain.pretrained.interfaces import Pretrained

class Encoder(Pretrained):
    # copied from the organizers' extract_voice_features_ecapa.py
    MODULES_NEEDED = ['compute_features', 'mean_var_norm', 'embedding_model']
    def encode_batch(self, wavs, wav_lens=None):
        if len(wavs.shape) == 1:
            wavs = wavs.unsqueeze(0)
        if wav_lens is None:
            wav_lens = torch.ones(wavs.shape[0], device=self.device)
        wavs, wav_lens = wavs.to(self.device).float(), wav_lens.to(self.device)
        feats = self.mods.compute_features(wavs)
        feats = self.mods.mean_var_norm(feats, wav_lens)
        return self.mods.embedding_model(feats, wav_lens)

ECAPA = Encoder.from_hparams(source='yangwang825/ecapa-tdnn-vox2', savedir=os.path.join(CACHE, 'ecapa_vox2'),
                             run_opts={'device': dev})
ECAPA.eval()

def enc_ecapa(paths, bs=16):
    # length-sorted batches + RELATIVE lengths (SpeechBrain convention), like the official script
    order = np.argsort([sf.info(p).duration for p in paths])
    out = np.zeros((len(paths), 192), np.float32)
    def run(idx):
        wavs = [torch.from_numpy(librosa.load(paths[j], sr=16000, mono=True)[0]) for j in idx]
        lens = torch.tensor([w.shape[0] for w in wavs], dtype=torch.float); L = int(lens.max())
        b = torch.zeros(len(wavs), L)
        for k, w in enumerate(wavs):
            b[k, :w.shape[0]] = w
        with torch.no_grad():
            return ECAPA.encode_batch(b, lens / L).squeeze(1).cpu().numpy()
    for i in tqdm(range(0, len(order), bs), desc='ecapa'):
        idx = order[i:i + bs]
        try:
            out[idx] = run(idx)
        except RuntimeError:          # out of memory on long clips -> one by one
            torch.cuda.empty_cache()
            for j in idx:
                out[j] = run([j])[0]
    return out

def enc_clip(paths):
    from transformers import CLIPModel, CLIPProcessor
    name = 'openai/clip-vit-large-patch14'
    model = CLIPModel.from_pretrained(name).half().to(dev).eval(); proc = CLIPProcessor.from_pretrained(name)
    out = []
    for i in tqdm(range(0, len(paths), 64), desc='clip'):
        ims = [Image.open(p).convert('RGB') for p in paths[i:i + 64]]
        x = proc(images=ims, return_tensors='pt')['pixel_values'].to(dev, torch.float16)
        with torch.no_grad():
            out.append(model.visual_projection(model.vision_model(pixel_values=x).pooler_output).float().cpu().numpy())
    return np.concatenate(out)

def enc_wavlm(paths, max_sec=10):
    from transformers import AutoFeatureExtractor, WavLMForXVector
    name = 'microsoft/wavlm-base-plus-sv'
    fe = AutoFeatureExtractor.from_pretrained(name); model = WavLMForXVector.from_pretrained(name).to(dev).eval()
    out = []
    for i in tqdm(range(0, len(paths), 32), desc='wavlm'):
        wavs = [librosa.load(p, sr=16000, mono=True)[0][:16000 * max_sec] for p in paths[i:i + 32]]
        x = fe(wavs, sampling_rate=16000, padding=True, return_attention_mask=True, return_tensors='pt').to(dev)
        with torch.no_grad():
            out.append(model(**x).embeddings.cpu().numpy())
    return np.concatenate(out)
print('ECAPA ready')

In [ ]:
# Cell 4 - CHECK: does our ECAPA extraction reproduce the official challenge voice features?
rng = np.random.default_rng(0)
rows = rng.choice(len(train_lines), 300, replace=False)
CHK = '/content/ecapa_check'; os.makedirs(CHK, exist_ok=True)
with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
    zi = zip_index(zf)
    paths = []
    for r in rows:
        member = zi[train_lines[r][2]]          # e.g. voices/English/id028/00000.wav
        paths.append(zf.extract(member, CHK))
mine = enc_ecapa(paths)
off = FEAT['ecapa']['train'][rows]
cos = (mine * off).sum(1) / (np.linalg.norm(mine, axis=1) * np.linalg.norm(off, axis=1) + 1e-9)
ECAPA_COMPATIBLE = bool(np.median(cos) > 0.98)
note('ECAPA check on 300 challenge clips: median cosine %.4f, min %.4f -> %s'
     % (np.median(cos), cos.min(), 'MATCHES official features' if ECAPA_COMPATIBLE else 'DIFFERENT - will re-extract ECAPA for challenge data too'))

In [ ]:
# Cell 5 - only if the check failed: extract our ECAPA for ALL challenge train + dev clips (consistency)
if not ECAPA_COMPATIBLE:
    p = os.path.join(FEATDIR, 'ecapa_ours.npz')
    if not os.path.exists(p):
        FULL = '/content/flag_data'; os.makedirs(FULL, exist_ok=True)
        saved = {}
        for z in ['train_set.zip', 'dev_set.zip']:
            if not os.path.exists(os.path.join(FULL, z + '.done')):
                shutil.copy(os.path.join(ZIP_DIR, z), '/content/' + z)
                with zipfile.ZipFile('/content/' + z) as zf:
                    zf.extractall(FULL)
                os.remove('/content/' + z); open(os.path.join(FULL, z + '.done'), 'w').close()
        def resolve(rel, list_path):
            q = os.path.dirname(list_path)
            while q.startswith(FULL):
                c = os.path.join(q, rel)
                if os.path.exists(c):
                    return c
                q = os.path.dirname(q)
        lists = {'train': ([p_ for p_ in [os.path.join(r, f) for r, _, fs in os.walk(FULL) for f in fs] if p_.endswith('train_English.txt')][0], 2)}
        for s in SPLIT_NAMES[1:]:
            track, lang = s.rsplit('_', 1)
            lp = [os.path.join(r, f) for r, _, fs in os.walk(FULL) for f in fs if f == lang + '_test.txt' and '/' + track + '/' in os.path.join(r, f)][0]
            lists[s] = (lp, 1)
        for s, (lp, col) in lists.items():
            wavs = [resolve(l.split()[col], lp) for l in open(lp) if l.strip()]
            saved[s] = enc_ecapa(wavs)
        np.savez(p, **saved)
    d = np.load(p); FEAT['ecapa'] = {s: d[s] for s in SPLIT_NAMES}
    note('Using OUR ECAPA features for the challenge data (consistent with MAV-Celeb).')
else:
    note('Using the OFFICIAL ECAPA features for the challenge data.')

In [ ]:
# Cell 6 - MAV-Celeb v1 + v3: read the train lists, unzip only the needed files, extract features (RESUMABLE)
EXT_PATH = os.path.join(FEATDIR, 'mavceleb_v1v3.npz')
SOURCES = [('v1', 'mavceleb_v1_train.zip', ['English_train.txt', 'Urdu_train.txt']),
           ('v3', 'mavceleb_v3_tran.zip', ['English_train.txt', 'German_train.txt'])]
if not os.path.exists(EXT_PATH):
    MAV = '/content/mav'; os.makedirs(MAV, exist_ok=True)
    spk, lang, voices, faces = [], [], [], []
    for ver, zname, lists in SOURCES:
        t0 = time.time()
        local_zip = '/content/' + zname
        shutil.copy(os.path.join(MAV_ZIPS, zname), local_zip)
        with zipfile.ZipFile(local_zip) as zf:
            zi = zip_index(zf)
            for lf in lists:
                n_missing = 0
                for l in open(os.path.join(MAV_LISTS, ver, lf)):
                    t = l.split()
                    if len(t) < 2:
                        continue
                    vm, fm = zi.get('voices/' + t[0]), zi.get('faces/' + t[1])
                    if vm is None or fm is None:
                        n_missing += 1; continue
                    voices.append(zf.extract(vm, os.path.join(MAV, ver)))
                    faces.append(zf.extract(fm, os.path.join(MAV, ver)))
                    spk.append(ver + '_' + t[0].split('/')[0]); lang.append(t[0].split('/')[1])
                print('%s %-18s ok, missing files: %d' % (ver, lf, n_missing))
        os.remove(local_zip)
        print('%s unpacked in %.1f min' % (ver, (time.time() - t0) / 60))
    note('MAV-Celeb rows: %d | people: %d | languages: %s'
         % (len(spk), len(set(spk)), dict(pd.Series(lang).value_counts())))
    ext = {'spk': np.array(spk), 'lang': np.array(lang)}
    for name, kind, fn in [('clip', faces, enc_clip), ('ecapa', voices, enc_ecapa), ('wavlm_sv', voices, enc_wavlm)]:
        part = EXT_PATH.replace('.npz', '_' + name + '.npy')
        if os.path.exists(part):
            ext[name] = np.load(part); continue
        t0 = time.time(); ext[name] = fn(kind).astype(np.float32); np.save(part, ext[name])
        note('MAV-Celeb %-8s extracted in %.1f min' % (name, (time.time() - t0) / 60))
    np.savez(EXT_PATH, **ext)
EXT = dict(np.load(EXT_PATH))
note('MAV-Celeb loaded: %d clips, %d people' % (len(EXT['spk']), len(set(EXT['spk']))))

In [ ]:
# Cell 7 - one combined table (challenge rows first, then MAV-Celeb rows) + model + validation (as Notebook 05)
import torch.nn as nn
import torch.nn.functional as Fn
from sklearn.metrics import roc_curve

def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def face_of(clip):
    return l2(clip).astype(np.float32)
def voice_of(ecapa, wavlm):
    return np.concatenate([l2(ecapa), l2(wavlm)], 1).astype(np.float32)

N_CH = len(y)
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(EXT['spk'])))}
Y_ALL = np.concatenate([y, np.array([ext_lab[s] for s in EXT['spk']])])
F_ALL = np.concatenate([face_of(FEAT['clip']['train']), face_of(EXT['clip'])])
V_ALL = np.concatenate([voice_of(FEAT['ecapa']['train'], FEAT['wavlm_sv']['train']), voice_of(EXT['ecapa'], EXT['wavlm_sv'])])
XF = torch.tensor(F_ALL, device=dev); XV = torch.tensor(V_ALL, device=dev)
EXT_ROWS = {'none': np.array([], dtype=int),
            'english': N_CH + np.where(EXT['lang'] == 'English')[0],
            'all': N_CH + np.arange(len(EXT['spk']))}

class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv)
        return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv)
        a = torch.sigmoid(self.att(torch.cat([f, v], 1)))
        fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused)
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t()
    same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm
    ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()   # HIGHER = same person

def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed)
    spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5
rng = np.random.default_rng(42)
folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k])
    va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
print('Combined table:', XF.shape, XV.shape, '| challenge rows', N_CH, '| MAV-Celeb rows', len(EXT['spk']))

In [ ]:
# Cell 8 - training (D_balanced recipe; a fixed number of steps per epoch so all variants cost the same)
SEEDS = [1, 2, 3]
P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60

def train_model(tr_idx, vsets=None, seed=1, max_epochs=MAX_EP):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    classes = np.unique(Y_ALL[tr_idx]); remap = {c: i for i, c in enumerate(classes)}
    ytr_np = np.array([remap[c] for c in Y_ALL[tr_idx]]); ytr = torch.tensor(ytr_np, device=dev)
    tr = torch.tensor(tr_idx, device=dev)
    norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6)   # TRAIN stats only
    mf, sf_, mv, sv = norm
    model = FOP(len(classes), XF.shape[1], XV.shape[1]).to(dev); model.norm = norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01)
    pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(classes))]
    best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(classes), P_ID, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv)
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j])
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[F], XV[V])) for V, F, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep

VARIANTS = {'E0_challenge_only': 'none', 'E1_plus_mav_english': 'english', 'E2_plus_mav_all': 'all'}
RESDIR = os.path.join(PROJECT, 'results'); os.makedirs(RESDIR, exist_ok=True)
CSV = os.path.join(RESDIR, 'nb06_cv.csv')
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['variant', 'seed', 'fold'])
t0 = time.time()
for name, ext_key in VARIANTS.items():
    for seed in SEEDS:
        for k in range(K):
            if ((done.variant == name) & (done.seed == seed) & (done.fold == k)).any():
                continue
            tr_idx, vsets = FOLD_DATA[k]
            model, best_ep = train_model(np.concatenate([tr_idx, EXT_ROWS[ext_key]]), vsets, seed=seed)
            e_std, e_gen = [eer(L, same_score(model, XF[F], XV[V])) for V, F, L in vsets]
            row = pd.DataFrame([dict(variant=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
            row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False)
            done = pd.concat([done, row])
            print('%-20s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %.0f min'
                  % (name, seed, k, best_ep, e_std, e_gen, (time.time() - t0) / 60))

In [ ]:
# Cell 9 - results table
R = pd.read_csv(CSV)
R['overall'] = (R['std'] + R['gender']) / 2
per_seed = R.groupby(['variant', 'seed'])[['std', 'gender', 'overall']].mean().reset_index()
tab = per_seed.groupby('variant').agg(std=('std', 'mean'), gender=('gender', 'mean'),
                                      overall=('overall', 'mean'), seed_sd=('overall', 'std'))
tab['gap'] = tab['gender'] - tab['std']
tab = tab.loc[[v for v in VARIANTS if v in tab.index]].round(2)
note('MORE PEOPLE? (K10 features, validation on held-out CHALLENGE people, 5-fold x 3 seeds)')
note(tab.to_string())

In [ ]:
# Cell 10 - final models (all 70 challenge people + MAV-Celeb, 3 seeds averaged) -> submission ZIPs for E1 and E2
LOWER_IS_SAME = True   # confirmed on CodaBench
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
for name in ['E1_plus_mav_english', 'E2_plus_mav_all']:
    ep = int(np.median(R[R.variant == name]['best_ep']))
    tr_all = np.concatenate([np.arange(N_CH), EXT_ROWS[VARIANTS[name]]])
    finals = [train_model(tr_all, None, seed=s, max_epochs=ep)[0] for s in SEEDS]
    OUT = '/content/sub_06_' + name
    for track in ['no_gender', 'gender']:
        os.makedirs(os.path.join(OUT, track), exist_ok=True)
        for lang, fname in LANG_FILE.items():
            split = track + '_' + lang
            pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang + '_test.txt')) if l.strip()]
            xf = torch.tensor(face_of(FEAT['clip'][split]), device=dev)
            xv = torch.tensor(voice_of(FEAT['ecapa'][split], FEAT['wavlm_sv'][split]), device=dev)
            assert len(pair_ids) == len(xf) == len(xv)
            s = np.mean([same_score(m, xf, xv) for m in finals], axis=0)
            out = -s if LOWER_IS_SAME else s
            with open(os.path.join(OUT, track, fname), 'w') as f:
                for pid, v in zip(pair_ids, out):
                    f.write('%s %.6f\n' % (pid, v))
    ZIP_PATH = os.path.join(SUBDIR, '06_' + name + '.zip')
    with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
        for track in ['no_gender', 'gender']:
            for fname in LANG_FILE.values():
                zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
    note('Submission ZIP saved: %s (%d epochs)' % (ZIP_PATH[len(BASE):], ep))

In [ ]:
# Cell 11 - experiment log + summary
LOG = os.path.join(PROJECT, 'experiments.csv')
log = pd.read_csv(LOG) if os.path.exists(LOG) else pd.DataFrame()
if len(log) and 'run' in log:
    log = log[~log.run.isin(['06_' + v for v in tab.index])]
new = pd.DataFrame([{'date': time.strftime('%Y-%m-%d %H:%M'), 'run': '06_' + v,
                     'change': 'K10 features + MAV-Celeb rows: %s' % VARIANTS[v], 'seed': '1,2,3',
                     'cv_std': tab.loc[v, 'std'], 'cv_gender': tab.loc[v, 'gender'],
                     'cv_overall': tab.loc[v, 'overall'], 'codabench_dev': ''} for v in tab.index])
pd.concat([log, new], ignore_index=True).to_csv(LOG, index=False)
print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)